# 05 — Untouched-model prompt and few-shot baselines

**Estimated time:** 50 minutes<br>
**Prerequisites:** 04 — Deterministic baselines<br>
**Learner-produced evidence:** a validation comparison of basic, strong, and few-shot prompts

## Learning objectives

- Hold model weights and validation examples constant across a prompt ladder.
- Build few-shot demonstrations exclusively from training records.
- Measure output quality and local resource use instead of eyeballing prose.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

Prompting changes model behavior without learning new weights, so it is usually cheaper and faster to test than fine-tuning. A strong prompt baseline tells you whether the base model already has the needed capability and whether the remaining gap is about instructions, examples, domain knowledge, or consistency. Prompt text and demonstrations are versioned system components.

## Key terms in plain language

- **prompt:** the instructions and context supplied to a model for one inference.
- **system message:** high-level behavior instructions supplied separately from user content in chat-style models.
- **zero-shot:** asking for a task without including solved examples in that request.
- **few-shot:** including a small number of solved demonstrations in the request; no weights are updated.
- **demonstration:** an example input and desired output placed in prompt context.
- **prompt baseline:** a versioned prompt strategy evaluated with unchanged model weights.
- **decoding:** the procedure and settings used to select output tokens from model scores.
- **output budget:** the maximum number of tokens the model is allowed to generate.


## Mental model — how to think about this

A prompt is a temporary program interpreted by a probabilistic runtime. Compare prompt variants like an A/B test: same base checkpoint, same inputs, same decoding settings, same parser and scorer; only the named prompt strategy changes. Basic, constrained, and few-shot prompts form another minimum-bar ladder before training weights.

### Running example

The same frozen base model sees the password message three ways: a basic request, a constrained request naming the JSON contract and allowed labels, and a few-shot request with training demonstrations. Only prompt context changes. Compare these as method versions, not as three anecdotes.

### Questions to ask before continuing

- Is the failure caused by missing task instructions, missing examples, missing knowledge, or unstable generation?
- Can the output space be made smaller and machine-checkable?
- Were all demonstrations selected without looking at frozen test examples or labels?
- Does a prompt improvement survive every relevant slice rather than a few showcased inputs?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Version the complete prompt contract.** Record system/user templates, label vocabulary, examples, decoding values, parser, base revision, and output budget.
- **Constrain outputs explicitly.** State allowed labels and schema, request only required fields, and validate rather than trusting prose assurances.
- **Source demonstrations from training data.** Select them with a documented rule and keep validation for choosing among prompt variants.
- **Make controlled comparisons.** Reuse inputs and evaluator; record any model, quantization, prompt, or inference-setting change separately.
- **Prefer the simplest prompt that meets the gate.** Longer context consumes memory and latency and can introduce contradictory instructions or accidental leakage.

## Common mistakes and why they fail

- **Cherry-picking impressive outputs.** A qualitative example illustrates behavior but cannot estimate a rate.
- **Using test examples as demonstrations.** That teaches the system from the exam and invalidates the comparison.
- **Changing several variables at once.** A new model plus new prompt plus different decoding cannot reveal which change caused the result.
- **Assuming few-shot must be better.** Poor, unrepresentative, or conflicting examples can reduce quality; measure it.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [MLflow prompt evaluation documentation](https://mlflow.org/docs/latest/genai/eval-monitor/running-evaluation/prompts/)
- **Tool guidance:** [MLflow evaluation-driven development overview](https://mlflow.org/docs/latest/genai/eval-monitor)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Construct prompts before loading a model

The basic prompt states the task. The strong prompt adds allowed labels,
output shape, train-derived category mapping, escalation guidance, and
safety constraints. Few-shot adds deterministic demonstrations from train.
No prompt is selected using frozen-test failures.


In [ ]:
import pandas as pd

from aai_local_finetuning.evaluation import (
    evaluate_predictions,
    recheck_evaluation_session,
    start_evaluation_session,
)
from aai_local_finetuning.learning import (
    generate_support_predictions,
    load_support_splits,
    report_row,
    select_few_shots,
    support_contract,
)
from aai_local_finetuning.modeling import LocalMLXPredictor, build_messages
from aai_local_finetuning.settings import load_settings

settings = load_settings()
splits = load_support_splits(settings, include_test=False)
allowed_intents, categories = support_contract(splits.train)
demonstrations = select_few_shots(splits.train, limit=4)
validation_example = splits.validation[0]

## Inspect the controlled change

The final user message stays identical. Only the instruction/context
changes. Demonstration IDs are not needed in the prompt, but selection is
deterministic from training records and can be reconstructed.


In [ ]:
prompt_ladder = {
    strategy: build_messages(
        validation_example.input_text,
        strategy=strategy,
        allowed_intents=list(allowed_intents),
        category_by_intent=categories,
        few_shot=demonstrations,
    )
    for strategy in ("basic", "strong", "few_shot")
}
{
    strategy: {
        "message_count": len(messages),
        "system_preview": messages[0]["content"][:240],
        "same_final_user_message": (
            messages[-1]["content"] == validation_example.input_text
        ),
    }
    for strategy, messages in prompt_ladder.items()
}

## Small measured validation experiment

This default selects one record from each of six distinct intents so Run
All remains practical on a MacBook Air. It is a stratified smoke probe,
not a representative validation estimate. Macro F1 is still calculated
over all 27 supported intents, so the 21 absent intents receive zero and
the absolute value is deliberately not a prompt-quality claim. Use probe
coverage plus obvious contract failures here; a broad validation run is
required before locking a winner.

One evaluation session spans the prompt comparison. Capturing it before
model inference and rechecking it after scoring binds every report to the
same governed source and exact package set.


In [ ]:
VALIDATION_LIMIT = 6
first_by_intent = {}
for record in splits.validation:
    first_by_intent.setdefault(record.target.intent, record)
    if len(first_by_intent) == VALIDATION_LIMIT:
        break
validation_probe = tuple(first_by_intent.values())
probe_context = {
    "selection": "one validation record per distinct intent",
    "records": len(validation_probe),
    "covered_intents": sorted(first_by_intent),
    "supported_intents": len(allowed_intents),
    "macro_f1_warning": (
        "Absent supported intents score zero; do not interpret this "
        "small-probe macro F1 as an absolute quality estimate."
    ),
}
prompt_evaluation_session = start_evaluation_session()
predictor = LocalMLXPredictor(settings.model_dir)
prompt_reports = {}
prompt_predictions = {}
for strategy in ("basic", "strong", "few_shot"):
    predictions = generate_support_predictions(
        predictor,
        validation_probe,
        strategy=strategy,
        train_records=splits.train,
        max_tokens=96,
    )
    prompt_predictions[strategy] = predictions
    prompt_reports[strategy] = evaluate_predictions(
        validation_probe,
        predictions,
        supported_intents=allowed_intents,
        evaluation_session=prompt_evaluation_session,
    )
recheck_evaluation_session(prompt_evaluation_session)
display(
    pd.DataFrame([report_row(name, report) for name, report in prompt_reports.items()])
)
probe_context

## Inspect output as evidence, not as a vibe

A raw preview helps diagnose format errors. The strict report—not visual
plausibility—determines JSON parse, schema validity, supported labels,
classification, response policy, latency, tokens, and memory. This course
currently records output tokens only; few-shot prompts also consume more
**input** tokens, so the displayed token field is not a total-cost comparison.
The first local generation can pay compilation/warm-up cost, which also
makes a six-record latency comparison diagnostic rather than definitive.


In [ ]:
[
    {
        "strategy": strategy,
        "example_id": predictions[0].example_id,
        "output_preview": predictions[0].raw_text[:300],
        "latency_ms": round(predictions[0].latency_ms, 1),
        "output_tokens": predictions[0].output_tokens,
    }
    for strategy, predictions in prompt_predictions.items()
]

## Exercise — lock a prompt strategy

Choose using validation evidence. Success means the rationale mentions
both classification and structured-output quality. Once notebook 07 is
opened, do not revise this choice in response to frozen-test errors.


In [ ]:
chosen_strategy = "strong"
choice_rationale = (
    "This is a conservative course default, not the computed winner. Use "
    "the constrained contract until a broad validation run compares all "
    "strategies; the small probe cannot establish few-shot superiority."
)
assert chosen_strategy in prompt_reports
{
    "chosen_strategy": chosen_strategy,
    "validation_metrics": report_row(chosen_strategy, prompt_reports[chosen_strategy]),
    "rationale": choice_rationale,
}

**Hint:** prefer a reproducible rule over choosing the nicest single
output. A six-example probe can reject obvious failures, not establish a
final winner.


## Checkpoint

You have three untouched-model baselines and a validation-locked prompt
choice. The base weights have not changed.

**Next:** `06_lora_finetuning.ipynb` inspects and optionally runs the
adapter change without using frozen-test evidence.
